# Toxicity re-check: cheating alone is not toxic

Adds one rule on top of the full codebook (`prompts/cheating_rule.txt`): a complaint about cheating is **not** toxicity by itself.
It still counts when the review also has insults, slurs, nationality stereotypes, threats, harassment, griefing or a toxic community.

**How to use: Run All twice.**
1. First with `SAMPLE = 100` (a random 100 of all reviews). Read the examples and the before/after table at the bottom.
2. If it looks right, change `SAMPLE = None` in section 2 and Run All again. It skips what is already saved and does the rest.

**Nothing from `scale_up.ipynb` is touched.** This notebook reads `data/gemini_labels_*.csv` and writes two NEW files:
`data/gemini_harm_recheck_*.jsonl` (cache) and `data/gemini_harm_recheck_*.csv` (result).
Only reviews labeled `harm_mention = yes` are asked again (a narrower rule can only turn yes into no).

## 1. Setup

In [ ]:
import os, sys, json, time, importlib
import pandas as pd
from dotenv import load_dotenv
from google import genai

sys.path.insert(0, "../scripts")
import harm_recheck
importlib.reload(harm_recheck)

load_dotenv("../.env")
if os.environ.get("USE_VERTEX") == "1":
    client = genai.Client(
        vertexai=True,
        project=os.environ["GOOGLE_CLOUD_PROJECT"],
        location=os.environ.get("GOOGLE_CLOUD_LOCATION", "global"),
    )
else:
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## 2. Choose the sample (the one thing to change)

In [ ]:
SAMPLE = 100   # <-- test first with 100. Then set to None and Run All again for everything.

GEMINI_MODEL = "gemini-3.5-flash-lite"
TEMPERATURE = 0
MODEL_TAG = GEMINI_MODEL.replace("/", "_")

SOURCE = f"../data/gemini_labels_{MODEL_TAG}.csv"         # read only, never written
CACHE  = f"../data/gemini_harm_recheck_{MODEL_TAG}.jsonl" # NEW
OUT    = f"../data/gemini_harm_recheck_{MODEL_TAG}.csv"   # NEW

src = pd.read_csv(SOURCE)
ok = src[src["status"] == "ok"].sample(frac=1, random_state=0).reset_index(drop=True)  # fixed shuffle: same order every run
scope = ok.head(SAMPLE) if SAMPLE else ok          # the reviews this run is about (all games mixed)
df_yes = scope[scope["harm_mention"] == "yes"].copy()
print(f"Scope: {len(scope):,} reviews ({'sample' if SAMPLE else 'ALL'}) -> {len(df_yes):,} are toxic and get re-checked")
scope["game"].value_counts()

## 3. Run

Every review is saved the moment it finishes. A second run skips what is saved.

In [ ]:
with open("../prompts/codebook_exact.txt", "r") as f:
    CODEBOOK = f.read()
with open("../prompts/cheating_rule.txt", "r") as f:
    RULE = f.read()

SYSTEM_RC = (
    "BACKGROUND. This is the full project codebook that produced the earlier labels.\n\n"
    + CODEBOOK
    + "\n\n==========\n"
    + RULE
)
print(f"Prompt: {len(SYSTEM_RC.splitlines())} lines")

run = harm_recheck.RecheckRun(
    df_yes, client, GEMINI_MODEL, SYSTEM_RC, CACHE,
    temperature=TEMPERATURE, workers=8,
)
run.start()

## 4. Save the result table (new file)

In [ ]:
out = run.export(OUT)
out["harm_mention_v2"].value_counts(dropna=False)

## 5. Read what changed

`turned to NO` = was toxic only because of cheating. `stayed YES` = still toxic. Check that both groups look right.

In [ ]:
pd.set_option("display.max_colwidth", 160)
done = out.dropna(subset=["harm_mention_v2"])
done = done[done["key"].isin(df_yes["key"])]
for title, mask in [("turned to NO (cheating only)", done["harm_mention_v2"] == "no"),
                    ("stayed YES", done["harm_mention_v2"] == "yes"),
                    ("uncertain", done["harm_mention_v2"] == "uncertain")]:
    s = done[mask]
    print(f"\n=== {title}: {len(s):,} ===")
    for _, r in s.sample(min(6, len(s)), random_state=1).iterrows():
        print("-", r["game"], "|", str(r["review"])[:170].replace("\n", " "))
        print("    before:", r["harm_reason"])
        print("    after :", r["harm_reason_v2"])

## 6. Before and after

The denominator is **every review in the scope** (not only the toxic ones). Reviews that were already `no` stay `no`.
With `SAMPLE = 100` this is an estimate from 100 random reviews. With `SAMPLE = None` it is the real number for all reviews.

In [ ]:
done = out[out["key"].isin(df_yes["key"])].dropna(subset=["harm_mention_v2"])
if len(done) < len(df_yes):
    print(f"WARNING: only {len(done):,} of {len(df_yes):,} toxic reviews are re-checked. Run all cells again.")

tot = scope["game"].value_counts()
before = scope[scope["harm_mention"] == "yes"]["game"].value_counts().reindex(tot.index).fillna(0)
after = done[done["harm_mention_v2"] == "yes"]["game"].value_counts().reindex(tot.index).fillna(0)
unc = done[done["harm_mention_v2"] == "uncertain"]["game"].value_counts().reindex(tot.index).fillna(0)

res = pd.DataFrame({
    "reviews": tot,
    "before: toxic %": (before / tot * 100).round(1),
    "after: toxic %": (after / tot * 100).round(1),
    "uncertain after %": (unc / tot * 100).round(1),
})
n = tot.sum()
res.loc["All three"] = [n, round(before.sum() / n * 100, 1), round(after.sum() / n * 100, 1), round(unc.sum() / n * 100, 1)]
res